# Preparando Dados

## Carregando e Lendo Dataset

In [ ]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fashionmnist' dataset.
Path to dataset files: /kaggle/input/fashionmnist


In [ ]:
print(os.listdir(path))

['t10k-labels-idx1-ubyte', 't10k-images-idx3-ubyte', 'fashion-mnist_test.csv', 'fashion-mnist_train.csv', 'train-labels-idx1-ubyte', 'train-images-idx3-ubyte']


## Separando Treino e teste

In [ ]:
import pandas as pd
train_df = pd.read_csv(os.path.join(path, 'fashion-mnist_train.csv'))
test_df = pd.read_csv(os.path.join(path, 'fashion-mnist_test.csv'))

print("treino: ", train_df.shape)
print("teste: ", test_df.shape)

treino:  (60000, 785)
teste:  (10000, 785)


In [ ]:
X_train = train_df.drop('label', axis=1)
y_train = train_df['label']

X_test = test_df.drop('label', axis=1)
y_test = test_df['label']

In [ ]:
print(X_train.shape)  # (60000, 784)
print(y_train.shape)  # (60000,)

(60000, 784)
(60000,)


In [ ]:
#Normalização
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_mn = scaler.fit_transform(X_train)
X_test_mn = scaler.transform(X_test)

#Base Line

Decision Tree Classifier utilizando todas as Features

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
clf_baseline = DecisionTreeClassifier(random_state=42)

clf_baseline.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

In [ ]:
pred = clf_baseline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 0.7963


Acurácia no conjunto de teste: 0.7943

Porcentagem de features selecionadas (em relação aos dados originais): 100%

Tempo de treinamento: 1min

Tempo necessário para encontrar o subconjunto de features: N/A (usaram-se todas)


# Wrapper
Forward Selection

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SequentialFeatureSelector

In [ ]:
# Tirando uma amostra do dataset
dt = DecisionTreeClassifier(random_state=42)

sfs = SequentialFeatureSelector(
    dt,
    n_features_to_select=50,
    direction='forward',
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

# IMPORTANTE: apenas dados de treinamento
sfs.fit(X_train, y_train)

# Features selecionadas
selected_features = X_train.columns[sfs.get_support()]

print(f"Quantidade de features selecionadas: {len(selected_features)}")


In [ ]:
X_train_fs = sfs.transform(X_train)
X_test_fs = sfs.transform(X_test)

n_original = X_train.shape[1]
n_selected = len(selected_features)

percentual = (n_selected / n_original) * 100

print(f"Features originais: {n_original}")
print(f"Features selecionadas: {n_selected}")
print(f"Percentual utilizado: {percentual:.2f}%")

clf = DecisionTreeClassifier(random_state=42)

clf.fit(X_train_fs, y_train)

pred = clf.predict(X_test_fs)

In [ ]:
percentual = 100 * 50 / 784
print(f"{percentual:.2f}% das features originais")

In [ ]:
# Modelo base
dt = DecisionTreeClassifier(random_state=42)

# Forward Selection
sfs = SequentialFeatureSelector(
    dt,
    n_features_to_select='auto',  # ou um número específico
    direction='forward',
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)

# IMPORTANTE: apenas dados de treinamento
sfs.fit(X_train, y_train)

# Features selecionadas
selected_features = X_train.columns[sfs.get_support()]

print(f"Quantidade de features selecionadas: {len(selected_features)}")



'\n# Forward Selection\nsfs = SequentialFeatureSelector(\n    dt,\n    n_features_to_select=\'auto\',  # ou um número específico\n    direction=\'forward\',\n    scoring=\'accuracy\',\n    cv=5,\n    n_jobs=-1\n)\n\n# IMPORTANTE: apenas dados de treinamento\nsfs.fit(X_train, y_train)\n\n# Features selecionadas\nselected_features = X_train.columns[sfs.get_support()]\n\nprint(f"Quantidade de features selecionadas: {len(selected_features)}")\n'

In [ ]:

X_train_fs = sfs.transform(X_train)
X_test_fs = sfs.transform(X_test)


'\nX_train_fs = sfs.transform(X_train)\nX_test_fs = sfs.transform(X_test)\n'

In [ ]:
n_original = X_train.shape[1]
n_selected = len(selected_features)

percentual = (n_selected / n_original) * 100

print(f"Features originais: {n_original}")
print(f"Features selecionadas: {n_selected}")
print(f"Percentual utilizado: {percentual:.2f}%")

NameError: name 'selected_features' is not defined

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

clf.fit(X_train_fs, y_train)

pred = clf.predict(X_test_fs)

NameError: name 'X_train_fs' is not defined

# Genetic Algorithm

In [ ]:
!pip install sklearn-genetic-opt

In [ ]:
import time
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn_genetic import GAFeatureSelectionCV

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

ga_estimator = GAFeatureSelectionCV(
    estimator=clf,
    cv=3,
    scoring="accuracy",
    population_size=10,
    generations=10,
    n_jobs=-1,
    verbose=True,
    keep_top_k=3,
    elitism=True,
)

timer_fs = time.process_time()
ga_estimator.fit(X_train, y_train)
timer_fe = time.process_time()

features = ga_estimator.support_

timer_ps = time.process_time()
y_predict_ga = ga_estimator.predict(X_test)
timer_pe = time.process_time()

accuracy = accuracy_score(y_test, y_predict_ga)

print('Acurácia final: \n' + str(accuracy))
print('Quantidade de features selecionadas: \n' + str(len(features)))
print('Tempo de Treinamento: \n' + str( (timer_pe - timer_ps) ) + ' segundos.')
print('Tempo de Escolha de features: \n' + str( (timer_fe - timer_fs) ) + ' segundos.')

/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


gen	nevals	fitness	fitness_std	fitness_max	fitness_min
0  	10    	0.78608	0.00140951 	0.78905    	0.78485    
1  	20    	0.787698	0.00123824 	0.7891     	0.785667   
2  	20    	0.788168	0.00101153 	0.789867   	0.7865     
3  	20    	0.788252	0.00104657 	0.789333   	0.785783   
4  	20    	0.788403	0.000665824	0.789333   	0.787483   
5  	20    	0.78808 	0.00116866 	0.789333   	0.786133   
6  	20    	0.788648	0.000853179	0.7899     	0.7876     
7  	20    	0.788865	0.00112236 	0.79       	0.786533   
8  	20    	0.789005	0.00095708 	0.79       	0.787433   
9  	20    	0.789052	0.00107097 	0.790167   	0.78675    
10 	20    	0.788578	0.00128731 	0.790167   	0.78675    


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but GAFeatureSelectionCV was fitted without feature names
  warnings.warn(


Acurácia final: 
0.7982
Quantidade de features selecionadas: 
784
Tempo de Treinamento: 
0.2984305609999751 segundos.
Tempo de Escolha de features: 
124.97259982900002 segundos.





Resultados do primeiro teste da bib: (Demorou aprox. 40 min com pop.size=10 e gen=10)
```
gen	nevals	fitness 	fitness_std	fitness_max	fitness_min
0  	10    	0.786125	0.00195769 	0.788717   	0.78145    
1  	20    	0.787605	0.00136043 	0.790217   	0.786067   
2  	20    	0.788953	0.0010432  	0.790217   	0.78715    
3  	20    	0.789122	0.00126724 	0.790783   	0.786083   
4  	20    	0.78914 	0.000991217	0.79045    	0.78755    
5  	20    	0.789102	0.00175186 	0.79045    	0.784817   
6  	20    	0.78951 	0.00119292 	0.79045    	0.78725    
7  	20    	0.78917 	0.00133807 	0.79045    	0.78735    
8  	20    	0.789203	0.00133101 	0.79045    	0.786917   
9  	20    	0.789738	0.00107858 	0.79045    	0.787467
```

Outro teste da bib:



```
gen	nevals	fitness	fitness_std	fitness_max	fitness_min
0  	10    	0.78608	0.00140951 	0.78905    	0.78485    
1  	20    	0.787698	0.00123824 	0.7891     	0.785667   
2  	20    	0.788168	0.00101153 	0.789867   	0.7865     
3  	20    	0.788252	0.00104657 	0.789333   	0.785783   
4  	20    	0.788403	0.000665824	0.789333   	0.787483   
5  	20    	0.78808 	0.00116866 	0.789333   	0.786133   
6  	20    	0.788648	0.000853179	0.7899     	0.7876     
7  	20    	0.788865	0.00112236 	0.79       	0.786533   
8  	20    	0.789005	0.00095708 	0.79       	0.787433   
9  	20    	0.789052	0.00107097 	0.790167   	0.78675    
10 	20    	0.788578	0.00128731 	0.790167   	0.78675

Acurácia final:
0.7982
Quantidade de features selecionadas:
784
Tempo de Treinamento:
0.2984305609999751 segundos.
Tempo de Escolha de features:
124.97259982900002 segundos.
```



## Genetic Algorithm

In [ ]:
import time
import numpy as np
import random

from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score

all_feature_names = clf_baseline.feature_names_in_

## Ferramentas

In [ ]:
# Calcular a acurácia antes de chamar
def fitness(clf: DecisionTreeClassifier, acc: float):
  """
  clf: Classificador a ser avaliado
  acc: Acurácia

  Calcular a acurácia antes de chamar!
  """
  f = clf.n_features_in_
  return ( (acc * 1000) - (f * 1.5))

In [ ]:
def mutate(genes: np.ndarray):
  random.seed()

  # Chance de mutação: 1/tamanho do gene
  if ((random.randint(0, (genes.size() -1) )) == 1):
    rng = np.random.default_rng()

    if (genes.size()  >= 100):
      new_genes_qt = int(genes.size() /100)
      new_genes = rng.choice(all_feature_names, size=new_genes_qt, replace=False)

      for i in range((genes.size() -1), (genes.size() - new_genes_qt - 1), -1):
        genes = np.delete(genes.size() , i)
      genes.append(new_genes)

      # para genes curtos
    else:
      new_genes = rng.choice(all_feature_names, size=1, replace=False)
      genes[genes.size()-1] = new_genes

  return genes

In [ ]:
def crossover(parentA: DecisionTreeClassifier, parentB: DecisionTreeClassifier):
  """
  Cria dois filhos utlizando crossover de um ponto.
  Os filhos são treinados com as features designadas.

  return:
  filho1: DecisionTreeClassifier,
  filho2: DecisionTreeClassifier
  """
  # Separação dos genes
  A_gene = np.split(parentA.feature_names_in_, 2)
  B_gene = np.split(parentB.feature_names_in_, 2)

  # Filho 1 -----------------------
  offspring1 = DecisionTreeClassifier(random_state=42)
  genes_offspring1 = np.concatenate(A_gene[0], B_gene[1])
  # Mutação
  mutate(genes_offspring1)
  X_train_1 = np.intersect1d(X_train, genes_offspring1)
  offspring1.fit(X_train_1, y_train)


  # Filho 2 -----------------------
  offspring2 = DecisionTreeClassifier(random_state=42)
  genes_offspring2 = np.concatenate(A_gene[1], B_gene[0])
  # Mutação
  mutate(genes_offspring2)
  X_train_2 = np.intersect1d(X_train, genes_offspring2)
  offspring1.fit(X_train_2, y_train)

  # Filhos treinados
  return offspring1, offspring2

In [ ]:
def generation(pop: int, candidates=None):
  """
  Caso não existam candidatos, cria e treina candidatos.
  Realiza os testes dos candidatos.

  pop: Tamanho da população, int
  candidates: Lista de candidatos DecisionTreeClassifier

  return:
  candidates: Lista de candidatos DecisionTreeClassifier
  fitness_score: Lista do fitness score dos candidatos, na mesma ordem.

  """
  fitness_score = []

  if (candidates):
    for i in range(len(candidates)-1):
      pred = candidates[i].predict(X_test)
      f = fitness(candidates[i], accuracy_score(y_test, pred))
      fitness_score.append(f)

  else:
    candidates = []
    for i in range(pop):
      clf = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
      candidates.append(clf)
      f = fitness(candidates[i], accuracy_score(y_test, pred))
      fitness_score.append(f)

  return candidates, fitness_score

## Algoritmo

## Execução